# Module 02 — Notebook 01: Integrated Gradients on Image Classification

## Learning Objectives
By completing this notebook, you will:
1. Apply Integrated Gradients to a pretrained ResNet-50 with three different baseline choices
2. Validate attribution quality using the completeness property
3. Visually compare zero, blurred, and random baselines side-by-side
4. Quantify baseline agreement using Spearman rank correlation
5. Apply NoiseTunnel to produce SmoothGrad-IG attributions

## Prerequisites
- Module 02 Guides 01 and 02
- Module 01 completed

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives(["— Notebook 01: Integrated Gradients on Image Classification", "Apply Integrated Gradients to a pretrained ResNet-50 with three different baseline choices", "Validate attribution quality using the completeness property", "Visually compare zero, blurred, and random baselines side-by-side", "Quantify baseline agreement using Spearman rank correlation", "Apply NoiseTunnel to produce SmoothGrad-IG attributions"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.transforms.functional import gaussian_blur
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
import urllib.request
import io
import json
from PIL import Image

from captum.attr import IntegratedGradients, NoiseTunnel
from captum.attr import visualization as viz

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(t.cpu() * std + mean, 0, 1)

def norm99(arr):
    v = np.percentile(np.abs(arr), 99)
    return np.clip(np.abs(arr), 0, v) / (v + 1e-8)

# Load model
model = models.resnet50(weights='IMAGENET1K_V1').to(DEVICE)
model.eval()
print("ResNet-50 loaded.")

## 1. Load Image and Get Prediction

In [ ]:
# Load ImageNet labels
LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)
try:
    with urllib.request.urlopen(LABELS_URL) as resp:
        labels = json.loads(resp.read().decode())
except Exception:
    labels = [f"class_{i}" for i in range(1000)]
    labels[208] = "Labrador retriever"; labels[281] = "tabby cat"

# Load sample image
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"
try:
    with urllib.request.urlopen(IMAGE_URL, timeout=10) as resp:
        raw = Image.open(io.BytesIO(resp.read())).convert('RGB')
except Exception:
    raw = Image.fromarray(np.random.randint(80, 180, (320, 320, 3), dtype=np.uint8))

input_tensor = preprocess(raw).unsqueeze(0).to(DEVICE)
img_np = denormalize(input_tensor.squeeze(0).cpu()).permute(1, 2, 0).numpy()

# Prediction
with torch.no_grad():
    probs = torch.softmax(model(input_tensor), dim=1)
    top_class = probs.argmax().item()
    top_prob = probs.max().item()

print(f"Prediction: {labels[top_class]} ({top_prob:.1%})")

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(img_np)
ax.set_title(f"Input: {labels[top_class]} ({top_prob:.1%})")
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Three Baselines: Side-by-Side Attribution

We compute IG attributions with three different baselines to show how the choice of baseline affects the attribution.

In [ ]:
ig = IntegratedGradients(model)

# Define the three baselines
baselines = {
    'Zero (black)':    torch.zeros_like(input_tensor),
    'Blurred':         gaussian_blur(input_tensor.cpu(), [41, 41], [15.0, 15.0]).to(DEVICE),
    'Random (avg 20)': torch.stack([
        torch.randn_like(input_tensor.cpu()) * 0.05 for _ in range(20)
    ]).mean(0).to(DEVICE),
}

# Compute IG for each baseline
baseline_attrs = {}
print("Computing IG attributions with three baselines...")
print(f"{'Baseline':<20} {'Delta':<12} {'Attribution Sum':<18} {'f(x)-f(x'')'}")
print("-" * 65)

for name, bl in baselines.items():
    inp = input_tensor.clone().requires_grad_(True)
    attr, delta = ig.attribute(
        inp, baselines=bl.to(DEVICE), target=top_class,
        n_steps=50, return_convergence_delta=True
    )

    # Completeness check
    with torch.no_grad():
        f_input = model(input_tensor)[0, top_class].item()
        f_baseline = model(bl.to(DEVICE))[0, top_class].item()

    attr_sum = attr.sum().item()
    expected = f_input - f_baseline

    print(f"{name:<20} {delta.item():<12.5f} {attr_sum:<18.4f} {expected:.4f}")

    baseline_attrs[name] = attr.detach().cpu()

print(f"\nDelta < 0.05 indicates good convergence.")
print(f"Attribution Sum ≈ f(x) - f(baseline) confirms completeness.")

In [ ]:
# Visualize: 3 baselines × 3 display modes
baseline_names = list(baseline_attrs.keys())
n_baselines = len(baseline_names)

fig, axes = plt.subplots(3, n_baselines, figsize=(5 * n_baselines, 14))
fig.suptitle(
    f"IG Attribution: Three Baseline Comparisons\n"
    f"Prediction: {labels[top_class]} ({top_prob:.1%})",
    fontsize=14
)

row_labels = ["Attribution Heatmap", "Overlay on Image", "Baseline Image"]

for col, name in enumerate(baseline_names):
    attr = baseline_attrs[name]
    attr_np = attr.squeeze(0).permute(1, 2, 0).numpy()
    attr_mag = norm99(np.abs(attr_np).mean(axis=-1))

    # Row 0: Heatmap
    im = axes[0, col].imshow(attr_mag, cmap='hot', vmin=0, vmax=1)
    axes[0, col].set_title(f"{name}\nHeatmap", fontsize=10)
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

    # Row 1: Overlay
    axes[1, col].imshow(img_np)
    axes[1, col].imshow(attr_mag, alpha=0.6, cmap='hot',
                         vmin=np.percentile(attr_mag, 80))
    axes[1, col].set_title(f"{name}\nOverlay", fontsize=10)
    axes[1, col].axis('off')

    # Row 2: The baseline itself
    bl = baselines[name]
    bl_disp = denormalize(bl.squeeze(0).cpu()).permute(1, 2, 0).numpy()
    axes[2, col].imshow(bl_disp)
    axes[2, col].set_title(f"{name}\nBaseline Image", fontsize=10)
    axes[2, col].axis('off')

# Add row labels
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, ha='right',
                             labelpad=10)

plt.tight_layout()
plt.show()

In [ ]:
# Quantify baseline agreement: Spearman rank correlation
print("Baseline Agreement: Spearman Rank Correlation")
print("(How similar are the attribution rankings across baselines?)")
print()

# Flatten attribution magnitudes for correlation
attr_vecs = {name: np.abs(attr.squeeze(0).permute(1,2,0).numpy()).mean(-1).flatten()
             for name, attr in baseline_attrs.items()}

names = list(attr_vecs.keys())
n = len(names)
corr_matrix = np.zeros((n, n))

for i, n1 in enumerate(names):
    for j, n2 in enumerate(names):
        rho, _ = scipy.stats.spearmanr(attr_vecs[n1], attr_vecs[n2])
        corr_matrix[i, j] = rho

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Spearman ρ')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
ax.set_yticklabels(names, fontsize=9)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{corr_matrix[i,j]:.2f}",
                ha='center', va='center', fontsize=11, fontweight='bold')
ax.set_title("Attribution Rank Agreement\nAcross Baseline Choices")
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  ρ > 0.7: baselines agree on which regions are important")
print("  ρ < 0.5: baselines focus on different aspects of the image")
print("\nAreas with high agreement across all three baselines are the most")
print("robust attributions — least dependent on baseline choice.")

## 3. Positive vs Negative Attributions

IG returns signed attributions. Positive means the feature increased the predicted class score; negative means it decreased it. Both are informative.

In [ ]:
# Use zero baseline for sign analysis
attr_zero = baseline_attrs['Zero (black)']
attr_np_signed = attr_zero.squeeze(0).permute(1, 2, 0).numpy()

# Separate positive and negative
attr_pos = np.maximum(attr_np_signed, 0).mean(axis=-1)
attr_neg = np.maximum(-attr_np_signed, 0).mean(axis=-1)

# Statistics
total_pos = attr_pos.sum()
total_neg = attr_neg.sum()
total_abs = total_pos + total_neg

print("Attribution Sign Analysis (Zero Baseline)")
print("=" * 50)
print(f"Total positive attribution: {total_pos:.4f} ({total_pos/total_abs:.1%} of total)")
print(f"Total negative attribution: {total_neg:.4f} ({total_neg/total_abs:.1%} of total)")
print(f"\nPositive = supports '{labels[top_class]}' prediction")
print(f"Negative = opposes '{labels[top_class]}' prediction")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f"Signed Attribution Analysis — {labels[top_class]}", fontsize=13)

axes[0].imshow(img_np)
axes[0].set_title("Input Image")
axes[0].axis('off')

im1 = axes[1].imshow(norm99(attr_pos), cmap='Greens')
axes[1].set_title("Positive Attribution\n(supports prediction)")
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(norm99(attr_neg), cmap='Reds')
axes[2].set_title("Negative Attribution\n(opposes prediction)")
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

# Combined: green=positive, red=negative
combined = np.zeros((*attr_pos.shape, 3))
combined[:, :, 1] = norm99(attr_pos)  # Green channel
combined[:, :, 0] = norm99(attr_neg)  # Red channel
axes[3].imshow(combined)
axes[3].set_title("Combined\n(Green=positive, Red=negative)")
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 4. SmoothGrad-IG vs Plain IG

NoiseTunnel applied to IG produces smoother, higher-quality attributions. The cost is approximately `nt_samples × n_steps` total passes.

In [ ]:
# Compare plain IG vs SmoothGrad-IG
baseline_zero = torch.zeros_like(input_tensor)

# Plain IG (50 steps)
inp = input_tensor.clone().requires_grad_(True)
plain_ig, delta_plain = ig.attribute(
    inp, baselines=baseline_zero, target=top_class,
    n_steps=50, return_convergence_delta=True
)
print(f"Plain IG delta: {delta_plain.item():.5f}")

# SmoothGrad-IG (10 samples × 50 steps = 500 passes)
nt = NoiseTunnel(ig)
inp = input_tensor.clone().requires_grad_(True)
smooth_ig = nt.attribute(
    inp,
    nt_type='smoothgrad',
    nt_samples=10,
    stdevs=0.05,
    baselines=baseline_zero,
    target=top_class,
    n_steps=50
)
print(f"SmoothGrad-IG: 10 samples × 50 steps = 500 total passes")

# Convert to display arrays
plain_np  = norm99(plain_ig.squeeze(0).cpu().permute(1,2,0).detach().abs().mean(-1).numpy())
smooth_np = norm99(smooth_ig.squeeze(0).cpu().permute(1,2,0).detach().abs().mean(-1).numpy())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Plain IG vs SmoothGrad-IG", fontsize=13)

axes[0].imshow(img_np)
axes[0].set_title("Input Image")
axes[0].axis('off')

axes[1].imshow(plain_np, cmap='hot')
axes[1].set_title("Plain IG (50 steps)")
axes[1].axis('off')

axes[2].imshow(smooth_np, cmap='hot')
axes[2].set_title("SmoothGrad-IG\n(10×50=500 passes)")
axes[2].axis('off')

# Difference map
diff = np.abs(plain_np - smooth_np)
im = axes[3].imshow(diff, cmap='Blues', vmin=0)
axes[3].set_title("Difference\n(Plain - SmoothGrad)")
axes[3].axis('off')
plt.colorbar(im, ax=axes[3], fraction=0.046)

plt.tight_layout()
plt.show()

# Quantify noise reduction
high_freq_plain  = (plain_np - np.mean(plain_np)).std()
high_freq_smooth = (smooth_np - np.mean(smooth_np)).std()
print(f"\nAttribution std (measure of spatial noise):")
print(f"  Plain IG:      {high_freq_plain:.4f}")
print(f"  SmoothGrad-IG: {high_freq_smooth:.4f}")
print(f"  Reduction: {(1 - high_freq_smooth/high_freq_plain):.1%}")

## Summary

### What You Demonstrated

1. **Baseline comparison:** Three baselines (zero, blurred, random) produce attribution maps that largely agree (Spearman ρ > 0.7) but differ in detail — reflecting their different interpretations
2. **Completeness validation:** For all three baselines, the sum of IG attributions matched $f(x) - f(x')$ with small convergence delta
3. **Signed attribution:** Positive and negative attributions both contain useful information about model reasoning
4. **SmoothGrad-IG:** Produces smoother, less noisy attribution maps at 10× the computational cost

### Key Takeaway

The zero baseline is the standard choice for image classification. Always verify convergence delta < 0.05 before trusting IG attributions.

### What's Next

**Notebook 02:** Token-level attribution for text classification using LayerIntegratedGradients on DistilBERT.